# Q2 — Decision Tree from Scratch using Gini Index

Builds a decision tree classifier using the Gini index as split criterion. The implementation is simple and intended for learning; it supports binary and categorical splits and continuous features by thresholding.

In [1]:
import pandas as pd
import math
from collections import Counter
from pprint import pprint

# Borrower dataset (from Figure 1)
data = {
    'Home Owner': ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'Marital Status': ['Single','Married','Single','Married','Divorced','Married','Divorced','Single','Married','Single'],
    'Annual Income': [125,100,70,120,95,60,220,85,75,90],
    'Defaulted Borrower': ['No','No','No','No','Yes','Yes','No','Yes','No','Yes']
}
df = pd.DataFrame(data)
df.head()

,Home Owner,Marital Status,Annual Income,Defaulted Borrower
0,Yes,Single,125,No
1,No,Married,100,No
2,No,Single,70,No
3,Yes,Married,120,No
4,No,Divorced,95,Yes


In [2]:
# Utility functions for Gini and splitting
def gini(groups, classes):
    # groups: list of lists of rows (as dicts)
    # classes: unique class labels
    n_instances = sum([len(g) for g in groups])
    gini_val = 0.0
    for group in groups:
        size = len(group)
        if size == 0:
            continue
        score = 0.0
        counts = Counter([row['Defaulted Borrower'] for row in group])
        for cls in classes:
            p = counts.get(cls, 0) / size
            score += p * p
        gini_val += (1.0 - score) * (size / n_instances)
    return gini_val

def test_split(index, value, dataset, feature):
    left, right = list(), list()
    # if numeric, use <= on value; if categorical, equality
    for row in dataset:
        val = row[feature]
        if isinstance(val, (int, float)):
            if val <= value:
                left.append(row)
            else:
                right.append(row)
        else:
            if val == value:
                left.append(row)
            else:
                right.append(row)
    return left, right

def get_split(dataset, features):
    class_values = list(set([row['Defaulted Borrower'] for row in dataset]))
    b_index, b_value, b_score, b_groups, b_feature = None, None, 999, None, None
    for feature in features:
        # generate candidate splits
        vals = sorted(list(set([row[feature] for row in dataset])))
        if all(isinstance(v, (int, float)) for v in vals):
            # try midpoints between consecutive values
            candidates = [(vals[i] + vals[i+1]) / 2.0 for i in range(len(vals)-1)]
        else:
            candidates = vals
        for val in candidates:
            groups = test_split(None, val, dataset, feature)
            gini_val = gini(groups, class_values)
            if gini_val < b_score:
                b_index, b_value, b_score, b_groups, b_feature = None, val, gini_val, groups, feature
    return {'feature': b_feature, 'value': b_value, 'groups': b_groups, 'gini': b_score}

def to_dict_rows(df):
    return df.to_dict(orient='records')

def majority_class(group):
    counts = Counter([row['Defaulted Borrower'] for row in group])
    return counts.most_common(1)[0][0]

# Build tree recursively
def build_tree(dataset, features, max_depth=10, min_size=1, depth=0):
    labels = [row['Defaulted Borrower'] for row in dataset]
    if len(set(labels)) == 1:
        return labels[0]
    if not features or depth >= max_depth or len(dataset) <= min_size:
        return majority_class(dataset)
    split = get_split(dataset, features)
    if not split['feature']:
        return majority_class(dataset)
    node = {'feature': split['feature'], 'value': split['value'], 'gini': split['gini']}
    left, right = split['groups']
    if not left or not right:
        return majority_class(dataset)
    node['left'] = build_tree(left, features, max_depth, min_size, depth+1)
    node['right'] = build_tree(right, features, max_depth, min_size, depth+1)
    return node

# Prepare dataset and features
records = to_dict_rows(df)
features = ['Home Owner', 'Marital Status', 'Annual Income']
tree = build_tree(records, features)
pprint(tree)

{'feature': 'Annual Income',
 'gini': 0.26666666666666666,
 'left': {'feature': 'Annual Income',
          'gini': 0.2222222222222222,
          'left': {'feature': 'Annual Income',
                   'gini': 0.0,
                   'left': 'Yes',
                   'right': 'No',
                   'value': 65.0},
          'right': 'Yes',
          'value': 80.0},
 'right': 'No',
 'value': 97.5}


You can use the produced tree by writing a simple `predict` function that traverses the tree. This implementation is educational and not optimized for large datasets.